# JalVaani v4.3: Pune fix, even intervals, width cap + the latest forecast for the dashboards

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

**Upload in Step 1** (where each file is on your laptop):

| file | what it is |
|---|---|
| `tft_model_v3.py` | `03_models/tft/`: the TFT network (unchanged) |
| `tft_v4.py` | `03_models/tft/`: the pipeline, **pipeline 4.7 or newer** (= v4.3) |
| `master_dataset_v4target.csv` | `data/master/`: the dataset (25 districts, 2023-01-01 → 2026-06-04) |
| `RESULTS_GUIDE.md` | `docs/`: the guide; it is copied into the results zip |
| `tft_metrics_per_district.csv` | *optional*: the OLD v1 file from `04_results/tft_prod_v1/`, for a side-by-side table |

**What v4.3 changes.** Post-processing only. The networks and LightGBM are the v4.1 ones; v4.2 showed they re-train bit-for-bit.
1. **Pune, rule G2.** Where LightGBM alone was more than 20% worse than seasonal-naive on a district's own past errors, that district's blend is refitted **without LightGBM**.
   - v4.2's guard replaced the whole blend and was rejected: it fired in 16 districts.
   - A dev-fold check of this narrower version found it fires for Pune and single-well Raigad only.
2. **Width cap.** A calibrated interval may reach at most 3 × the network's own distance from the median. This removes the rare 90–108 m intervals.
3. **Even intervals, rule D2.** When choosing raw or calibrated per model and horizon, the share of districts with honest coverage (65–90%) now counts, not only the pooled coverage.
4. **The latest forecast.** A forecast from the last groundwater reading, made exactly like an evaluated fold, plus `dashboard_export/`, the files the dashboards read.

**Honesty note.** Changes 2 and 3 were designed after looking at v4.2's results, including the locked folds. So v4.3's locked-fold numbers are a refinement check, not a fresh hold-out. All choices below still use dev folds 1–4 only.

**What it does (about 40 min):** train dev folds once → rules G2 and D2 on dev folds → final run (trains folds 5–6) → v4.2 post-processing on the same networks as the reference → latest forecast → simple results and dashboard export → downloads `tft_v4_3_outputs.zip`.

In [ ]:
# ── STEP 1: upload the files listed above (from your laptop) ─────────────────
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install -q lightgbm

In [ ]:
# ── STEP 2: setup ────────────────────────────────────────────────────────────
import sys, time, shutil, zipfile, os, json
from pathlib import Path
import numpy as np, pandas as pd, torch
from IPython.display import Image, display, Markdown
sys.path.insert(0, '/content')
import tft_v4 as v4

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
assert tuple(map(int, v4.PIPELINE_VERSION.split('.'))) >= (4, 7), \
    f'old tft_v4.py ({v4.PIPELINE_VERSION}) - upload the v4.3 one (pipeline 4.7)'
print('GPU:', torch.cuda.get_device_name(0), '| pipeline', v4.PIPELINE_VERSION)

DATA = '/content/master_dataset_v4target.csv'
OUT  = Path('/content/outputs/v4_3')
OUT.mkdir(parents=True, exist_ok=True)
assert Path(DATA).exists(), 'master_dataset_v4target.csv was not uploaded'
T0 = time.time()
def elapsed(): return (time.time() - T0) / 60
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

In [ ]:
# ── STEP 3: settings ─────────────────────────────────────────────────────────
SEEDS = [42, 1337, 2026, 7, 99, 123, 555, 777, 2024, 31337]          # the v4.1 / v4.2 final seeds

# networks and LightGBM: the v4.1 final set-up (unchanged)
REF_KW = dict(target_mode='snaive', calib_mode='v4', combine='stack',
              train_stride=1, epoch_frac=1/7, use_refit=False,
              tab_features='v2', tab_stride=1, tab_fresh=True, gbm_n_estimators=400, gbm_lr=0.03, gbm_min_child=100,
              stack_prior_n=500)
# the published v4.2 post-processing (static conformal; reported intervals chosen by v4.2's rule D)
V42_POST = dict(aci_feedback='cell', aci_gamma=0.0, aci_alpha_min=0.01, aci_alpha_max=0.30,
                interval_choice={'TFT': {'t+7': 'raw', 't+14': 'raw', 't+30': 'raw'},
                                 'Ensemble': {'t+7': 'raw', 't+14': 'raw', 't+30': 'calibrated'}})
# v4.3 calibration: v4.2's static conformal + the width cap (3, fixed in advance, not tuned)
V43_CAL = dict(aci_feedback='cell', aci_gamma=0.0, aci_alpha_min=0.01, aci_alpha_max=0.30, interval_width_cap=3.0)

def make(tag, mode='dev', **kw):
    cfg = v4.Config(tag=tag, mode=mode, data_path=DATA, out_root=str(OUT), seeds=list(SEEDS))
    for k, v in {**REF_KW, **kw}.items():
        setattr(cfg, k, v)
    return cfg

def run_dev(tag, **kw):
    # dev folds 1-4 only; after the first call the networks and LightGBM come from the cache
    return v4.run(make(tag, mode='dev', **kw), verbose=False)

def brief(tag, res):
    S = v4.report(res, verbose=False, save_predictions=False)
    rows = []
    for name, sm in S.items():
        h, c = sm['head'], sm['cov']
        rows.append({'run': tag, 'model': name,
                     **{f'vs no-change {k}': h.loc[k, 'rMAE_P'] for k in v4.HORDER},
                     **{f'vs seasonal {k}': h.loc[k, 'rMAE_SN'] for k in v4.HORDER},
                     **{f'cov {k}': c.loc[k, 'Cov (%)'] for k in v4.HORDER},
                     **{f'interval score {k}': c.loc[k, 'Interval score (m)'] for k in v4.HORDER},
                     **{f'districts 65-90 {k}': c.loc[k, 'districts in 65-90% (%)'] for k in v4.HORDER},
                     'mean vs seasonal': h['rMAE_SN'].mean()})
    return pd.DataFrame(rows)

## Step A: train dev folds 1–4 once (v4.1 networks, 10 seeds; v4.2 post-processing)
Every variant below reuses these networks and LightGBM models from the cache. Folds 5–6 are not computed until the final run.

In [ ]:
print(f'Training dev folds 1-4 with {len(SEEDS)} seeds ...', flush=True)
REF_DEV = run_dev('dev_ref', **V42_POST)
B_REF = brief('ref = v4.2 post-processing', REF_DEV)
display(B_REF.round(3))
print('Check: the v4.2 run had dev mean Ensemble vs seasonal 0.9501 (rule G table); expect the same value')
print(f'[{elapsed():.0f} min elapsed]')

## Rule G2: the LightGBM-only guard (Pune)
The guard works per district and horizon, on that district's own pool of past out-of-sample errors (at least 50 weighted rows). If LightGBM alone was more than 20% worse than seasonal-naive alone there, the district's blend is refitted without LightGBM, still pulled toward the overall weights.

**Adopt it if, on dev folds 1–4 (21 headline districts):**
- the mean Ensemble error ÷ seasonal-naive error over t+7, t+14 and t+30 does **not get worse**; **and**
- no district's Ensemble MASE rises by more than **0.02** at any horizon.

In [ ]:
GUARD_DEV = run_dev('dev_lgbm_guard', **V42_POST, lgbm_guard=True)
ADOPT_GUARD, m_off, m_on, GTAB = v4.guard_decision(REF_DEV, GUARD_DEV)
GTAB.to_csv(OUT / 'guard_decision.csv', index=False)
if len(GUARD_DEV['guarded']):
    GUARD_DEV['guarded'].to_csv(OUT / 'guard_cases_dev.csv', index=False)
print(f'dev mean Ensemble vs seasonal: guard off {m_off:.4f} | on {m_on:.4f} | '
      f'largest district MASE rise {GTAB["change"].max():+.3f}')
display(Markdown('**Guarded (district, horizon, fold) on the dev folds**')); display(GUARD_DEV['guarded'])
display(Markdown('**Pune, Ensemble MASE on the dev folds**')); display(GTAB[GTAB.district == 'Pune'])
print('ADOPT lgbm_guard:', ADOPT_GUARD)
GUARD_KW = {'lgbm_guard': ADOPT_GUARD}

## Rule D2: report raw or calibrated, per model × horizon (evenness-aware)
The calibration is v4.2's static conformal plus the width cap (3×), both fixed in advance. On dev folds 1–4:
1. keep the options whose pooled coverage is within **75–85%**;
2. drop any whose interval score is more than **5%** worse than the best of those;
3. of the rest, take the one with the larger share of districts covered **65–90%**; within 5 points, the lower interval score, then the un-calibrated one;
4. if neither is within 75–85%, take the lower interval score.

In [ ]:
CAP_DEV = run_dev('dev_cap', **V43_CAL, **GUARD_KW)
ITAB = v4.interval_table(CAP_DEV)
INTERVAL_CHOICE, DECIDE = v4.pick_interval_choice_v2(ITAB)
ITAB.to_csv(OUT / 'interval_choice_table.csv', index=False)
DECIDE.to_csv(OUT / 'interval_choice_decisions.csv', index=False)
display(ITAB.round(3)); display(DECIDE.round(3))
FINAL_KW = {**V43_CAL, **GUARD_KW, 'interval_choice': INTERVAL_CHOICE}
print('Final v4.3 post-processing:', FINAL_KW, f'  [{elapsed():.0f} min elapsed]')

## Final run: all 6 folds
This trains folds 5–6; folds 1–4 come from the cache. Results are reported for all 6 folds and for folds 5–6 (locked). The v4.2 post-processing on the same networks is computed too, at no cost.

In [ ]:
FINAL_TAG = 'FINAL_v4_3'
res = v4.run(make(FINAL_TAG, mode='all', **FINAL_KW), verbose=True)
print('\n\n================  ALL 6 FOLDS  ================')
S_all = v4.report(res, verbose=True)

In [ ]:
print('================  LOCKED TEST: folds 5-6 (Oct 2025 -> Jun 2026)  ================')
locked = v4.subset_folds(res, list(v4.LOCKED_FOLDS), FINAL_TAG + '_locked')
S_locked = v4.report(locked, verbose=True, save_predictions=False)
QUOTE = v4.results_to_quote(S_locked, S_all)
QUOTE.to_csv(OUT / 'RESULTS_TO_QUOTE.csv', index=False)

# the v4.2 post-processing on the SAME networks (no training: everything is cached)
ref = v4.run(make('REF_v4_2_postproc', mode='all', **V42_POST), verbose=False)
R_all = v4.report(ref, verbose=False, save_predictions=False)
R_locked = v4.report(v4.subset_folds(ref, list(v4.LOCKED_FOLDS), 'REF_v4_2_postproc_locked'),
                     verbose=False, save_predictions=False)
REFQ = v4.results_to_quote(R_locked, R_all)
REFQ.to_csv(OUT / 'RESULTS_TO_QUOTE_ref_v4_2_postproc.csv', index=False)

# published v4.2 numbers (Colab run of 2026-10-30): MAE, vs no-change, vs seasonal, coverage, interval score
V42 = pd.DataFrame([
    (t, m, h, *vals) for (t, m), rows in {
        ('locked folds 5-6', 'TFT'): [(0.3264, 0.8725, 0.9756, 80.0541, 1.7781), (0.4276, 0.8034, 0.9713, 78.4623, 2.2087), (0.6023, 0.6746, 0.9597, 77.2541, 2.8876)],
        ('locked folds 5-6', 'Ensemble'): [(0.314, 0.8392, 0.9384, 81.6281, 1.7311), (0.4059, 0.7627, 0.9221, 80.5569, 2.1223), (0.5501, 0.6162, 0.8766, 79.0528, 3.1089)],
        ('all 6 folds', 'TFT'): [(0.444, 0.9346, 1.0003, 79.7482, 2.4913), (0.6286, 0.8764, 1.0018, 77.1754, 3.4211), (0.9236, 0.764, 0.9758, 73.0, 4.7753)],
        ('all 6 folds', 'Ensemble'): [(0.4291, 0.9031, 0.9666, 80.8527, 2.4429), (0.5953, 0.8301, 0.9488, 79.0833, 3.2932), (0.8605, 0.7118, 0.9092, 77.0985, 5.2114)],
    }.items() for h, vals in zip(v4.HORDER, rows)],
    columns=['test set', 'model', 'horizon', 'v4.2 MAE (m)', 'v4.2 vs no-change', 'v4.2 vs seasonal', 'v4.2 coverage',
             'v4.2 interval score'])
cols = {'MAE (m)': 'MAE (m)', 'rMAE vs persistence': 'vs no-change', 'rMAE vs seasonal': 'vs seasonal',
        'coverage 80% (%)': 'coverage', 'interval score (m)': 'interval score', 'interval': 'interval'}
def tagged(q, pre):
    return q[['test set', 'model', 'horizon'] + list(cols)].rename(columns={k: f'{pre} {v}' for k, v in cols.items()})
CMP = V42.merge(tagged(REFQ, 'same-net v4.2-post'), on=['test set', 'model', 'horizon']) \
         .merge(tagged(QUOTE, 'v4.3'), on=['test set', 'model', 'horizon'])
CMP['v4.3 - same-net: vs no-change'] = (CMP['v4.3 vs no-change'] - CMP['same-net v4.2-post vs no-change']).round(4)
CMP['v4.3 - same-net: interval score'] = (CMP['v4.3 interval score'] - CMP['same-net v4.2-post interval score']).round(3)

# share of headline districts with coverage 65-90% (the evenness v4.3 targets)
def evenness(S, label):
    return pd.DataFrame([{'run': label, 'model': m, 'horizon': h, 'districts 65-90 (%)': S[m]['cov'].loc[h, 'districts in 65-90% (%)']}
                         for m in S for h in v4.HORDER])
EVEN = pd.concat([evenness(R_all, 'same-net v4.2-post, all folds'), evenness(S_all, 'v4.3, all folds'),
                  evenness(R_locked, 'same-net v4.2-post, locked'), evenness(S_locked, 'v4.3, locked')])
EVEN = EVEN.pivot_table(index=['model', 'horizon'], columns='run', values='districts 65-90 (%)').round(0)
CMP.to_csv(OUT / 'v4_2_vs_v4_3.csv', index=False); EVEN.to_csv(OUT / 'district_evenness_v4_2_vs_v4_3.csv')
display(Markdown('### v4.2 → v4.3 (lower ratios and interval scores are better)')); display(CMP.round(3))
display(Markdown('### Districts with honest coverage (65–90%), % of 21')); display(EVEN)

# very wide intervals (all districts): reported Ensemble / TFT intervals wider than 20 m
def n_wide(r, model):
    rr = v4.reported_rows(r['daily'], model, r['cfg']); return int(((rr.q90 - rr.q10) > 20).sum())
WIDE = pd.DataFrame([{'model': m, 'same-net v4.2-post': n_wide(ref, m), 'v4.3': n_wide(res, m)} for m in ('Ensemble', 'TFT')])
WIDE.to_csv(OUT / 'wide_intervals_v4_2_vs_v4_3.csv', index=False)
display(Markdown('### Reported intervals wider than 20 m (all districts, all folds)')); display(WIDE)

# Pune, Ensemble MASE (all folds): published v4.2 1.000 / 1.050 / 1.128
PUNE = pd.concat([R_all['Ensemble']['per_dist'].assign(run='same-net v4.2-post'),
                  S_all['Ensemble']['per_dist'].assign(run='v4.3')])
PUNE = PUNE[PUNE.district == 'Pune'][['run', 'Horizon', 'rMAE_P', 'rMAE_SN', 'MAE (m)']]
PUNE.to_csv(OUT / 'pune_v4_2_vs_v4_3.csv', index=False)
display(Markdown('### Pune, Ensemble, all folds (published v4.2 MASE: 1.000 / 1.050 / 1.128)')); display(PUNE.round(3))
display(Markdown('### Results to quote (v4.3)')); display(QUOTE.round(3))

## The latest forecast (for the dashboards)
This is a forecast from each district's **last groundwater reading**. It is made exactly like an evaluated fold:
- the networks train up to three months before the issue date, and early-stop and calibrate on the last three months;
- LightGBM is refitted up to the issue date;
- the post-processing uses the errors of all six evaluated folds.

There are no actuals for it yet. When CGWB publishes newer readings, re-running this notebook with the new master moves the forecast forward.

In [ ]:
LATEST = v4.forecast_latest(res['cfg'], res['pool'])
v4.export_dashboard(res, LATEST, OUT / 'dashboard_export')
t = LATEST['table']
display(Markdown(f"**Issued from:** {t['issue_date'].min().date()} … {t['issue_date'].max().date()}  ·  "
                 f"targets up to {t['target_date'].max().date()}"))
show = t[t.model == 'Ensemble'].assign(change_m=lambda x: (x.q50 - x.level_now).round(2))
display(show[['district', 'tier', 'horizon', 'level_now', 'q10', 'q50', 'q90', 'change_m', 'interval']].round(2))
print(f'[{elapsed():.0f} min elapsed]')

## Simple results (v1 style): all 24 districts, Ensemble and TFT (reported intervals)

In [ ]:
SIMPLE = OUT / 'simple'
for model in ('Ensemble', 'TFT'):
    v4.export_simple(res, model, SIMPLE / model.lower())
v4.export_shared(res, SIMPLE)
if Path('/content/tft_metrics_per_district.csv').exists():
    v4.compare_v1('/content/tft_metrics_per_district.csv', SIMPLE, label='v4.3')
    print('v1 comparison written (NOT like-for-like - see its caveat column)')
if Path('/content/RESULTS_GUIDE.md').exists():
    shutil.copy('/content/RESULTS_GUIDE.md', OUT / 'RESULTS_GUIDE.md')
else:
    print('RESULTS_GUIDE.md was not uploaded - the zip will not contain the guide')

lines = ['# JalVaani v4.3 run summary', '',
         f'- pipeline {v4.PIPELINE_VERSION}, data {Path(DATA).name} (md5 {v4.file_md5(DATA)[:8]})',
         f'- networks / LightGBM: the v4.1 final set-up {REF_KW}',
         f'- seeds: {SEEDS}', f'- total run time: {elapsed():.0f} min',
         '- rules G2 and D2 used dev folds 1-4 only; the width cap (3) was fixed in advance',
         '- v4.3 was designed after seeing v4.2 results incl. folds 5-6: its locked numbers are a refinement check', '',
         f'## Rule G2 (LightGBM-only guard): adopted = {ADOPT_GUARD}',
         f'dev mean Ensemble vs seasonal: off {m_off:.4f}, on {m_on:.4f}; largest district rise {GTAB["change"].max():+.3f}', '',
         '## Rule D2 (reported interval per model and horizon)', '```', DECIDE.round(3).to_string(index=False), '```', '',
         '## v4.2 -> v4.3', '```', CMP.round(3).to_string(index=False), '```', '',
         '## Districts with coverage 65-90% (% of 21)', '```', EVEN.to_string(), '```', '',
         '## Reported intervals wider than 20 m', '```', WIDE.to_string(index=False), '```', '',
         '## Pune (Ensemble, all folds)', '```', PUNE.round(3).to_string(index=False), '```', '',
         f"## Latest forecast: issued {LATEST['table']['issue_date'].max().date()}, "
         f"{LATEST['table']['district'].nunique()} districts (dashboard_export/)"]
(OUT / 'RUN_SUMMARY.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'done   [{elapsed():.0f} min elapsed]')

In [ ]:
# ── the key results, shown here ──────────────────────────────────────────────
display(Markdown('### Reported interval per model and horizon')); display(DECIDE[['model', 'horizon', 'pick', 'reason']])
for model in ('ensemble', 'tft'):
    display(Markdown(f'### {model.upper()}: headline (v1-style)'))
    display(pd.read_csv(SIMPLE / model / f'{model}_metrics_headline.csv').round(3))
for f in ['ensemble/ensemble_coverage_heatmap.png', 'tft/tft_coverage_heatmap.png',
          'ensemble/ensemble_per_district_mase_t30.png', 'ensemble/quantile/quantile_pune.png']:
    p = SIMPLE / f
    if p.exists():
        display(Markdown(f'**{f}**')); display(Image(filename=str(p), width=1100))

In [ ]:
# ── zip everything except the model caches, and download ────────────────────
# the dev / reference runs reuse the final run's networks, so only FINAL_v4_3 keeps its .pt files
zpath = '/content/tft_v4_3_outputs.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and '_cache' not in p.parts and (p.suffix != '.pt' or FINAL_TAG in p.parts):
            z.write(p, p.relative_to(OUT))
print(f'{zpath}: {os.path.getsize(zpath)/1e6:.1f} MB')
files.download(zpath)